# Two-arm render branch comparison

这个 notebook 单独对比两条 renderer 分支：
- A 分支：共享 [`cam_to_base`](../airexo/airexo/helpers/renderer.py:158)（严格模拟 [`CalibrationInfo.get_camera_to_base()`](../airexo/airexo/calibration/calib_info.py:81) + [`RobotRenderer`](../airexo/airexo/helpers/renderer.py:150)）
- B 分支：左右分离 [`cam_to_left_base`](../airexo/airexo/helpers/renderer.py:277) / [`cam_to_right_base`](../airexo/airexo/helpers/renderer.py:278)（严格模拟 [`SeparateRobotRenderer`](../airexo/airexo/helpers/renderer.py:269)）

所有需要的路径和变量都在前面统一声明，不依赖别的 notebook cell。

In [1]:
import os
import sys
import json
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R
from omegaconf import OmegaConf

WORKSPACE_ROOT = '/home/haoxiang/rise2_mask_aware'
AIREXO_ROOT = os.path.join(WORKSPACE_ROOT, 'airexo')
for p in [WORKSPACE_ROOT, AIREXO_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

for _m in [m for m in list(sys.modules.keys()) if m == 'airexo' or m.startswith('airexo.') or m == 'mask' or m.startswith('mask.')]:
    del sys.modules[_m]

from airexo.helpers.constants import (
    ROBOT_PREDEFINED_TRANSFORMATION,
    ROBOT_LEFT_REAL_BASE_TO_REAL_BASE,
    ROBOT_RIGHT_REAL_BASE_TO_REAL_BASE,
)
from airexo.helpers.rotation import average_xyz_rot_quat
from airexo.helpers.renderer import SeparateRobotRenderer
from mask.renderer import ArmOnlyRobotRenderer


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/home/haoxiang/miniforge3/envs/rise2/lib/python3.10/site-packages/transformations/transformations.py:1924: UserWarning: No module named 'transformations._transformations'
  warnings.warn(str(exc))


In [ ]:
# =========================
# 用户集中修改区
# =========================
TASK_ROOT = Path('/data/haoxiang/data/task0012_260321/task0012_toys_basket_converted')
CALIB_DIR = Path('/data/haoxiang/data/task0012_260321/calib')
LEFT_JSON = Path('/data/haoxiang/data/task0012_260321/calib/left_global_20260104/result.json')
RIGHT_JSON = Path('/data/haoxiang/data/task0012_260321/calib/right_global_20260104/result.json')
FAKE_CALIB_TIMESTAMP = '20260104_fake_airexo_robot'
SCENE_NAME = 'scene_0001'
FRAME_STEM = '1774059832169'

LEFT_CFG_PATH = os.path.join(WORKSPACE_ROOT, 'airexo/airexo/configs/joint/left/robot.yaml')
RIGHT_CFG_PATH = os.path.join(WORKSPACE_ROOT, 'airexo/airexo/configs/joint/right/robot.yaml')
ROBOT_INHAND_URDF = os.path.join(WORKSPACE_ROOT, 'airexo/airexo/urdf_models/robot/robot_inhand.urdf')
LEFT_URDF = os.path.join(WORKSPACE_ROOT, 'airexo/airexo/urdf_models/robot/left_robot.urdf')
RIGHT_URDF = os.path.join(WORKSPACE_ROOT, 'airexo/airexo/urdf_models/robot/right_robot.urdf')

# 如果希望直接手填 joint，可以把 USE_LOWDM_GRIPPER_FOR_LAST_JOINT 改成 False 后手动设置
USE_LOWDM_GRIPPER_FOR_LAST_JOINT = True
LEFT_JOINT_MANUAL = np.zeros(8, dtype=np.float32)
RIGHT_JOINT_MANUAL = np.zeros(8, dtype=np.float32)


In [ ]:
def pose7_wxyz_to_mat(pose7):
    pose7 = np.asarray(pose7, dtype=np.float64).reshape(7)
    t = pose7[:3]
    qw, qx, qy, qz = pose7[3:]
    T = np.eye(4, dtype=np.float64)
    T[:3, :3] = R.from_quat([qx, qy, qz, qw]).as_matrix()
    T[:3, 3] = t
    return T

def load_pose_in_link(json_path):
    data = json.loads(Path(json_path).read_text())
    return pose7_wxyz_to_mat(np.asarray(data['pose_in_link'], dtype=np.float64))

def read_rgb(rgb_path):
    rgb_bgr = cv2.imread(str(rgb_path), cv2.IMREAD_COLOR)
    assert rgb_bgr is not None, rgb_path
    return cv2.cvtColor(rgb_bgr, cv2.COLOR_BGR2RGB)

def overlay_render(rgb, render, mask, alpha=0.6):
    overlay = rgb.copy()
    overlay[mask > 0] = (alpha * render[mask > 0] + (1 - alpha) * overlay[mask > 0]).astype(np.uint8)
    return overlay


In [ ]:
scene_path = TASK_ROOT / 'train' / SCENE_NAME
fake_calib_path = CALIB_DIR / f'{FAKE_CALIB_TIMESTAMP}.npy'
assert fake_calib_path.exists(), fake_calib_path

fake_calib = np.load(fake_calib_path, allow_pickle=True).item()
camera_serial = list(fake_calib['camera_serials_global'])[0]
intrinsic = np.asarray(fake_calib['intrinsics'][camera_serial], dtype=np.float32)

cam_path = scene_path / f'cam_{camera_serial}'
lowdim_path = scene_path / 'lowdim' / f'{FRAME_STEM}.npy'
rgb_path = cam_path / 'color' / f'{FRAME_STEM}.png'
depth_path = cam_path / 'depth' / f'{FRAME_STEM}.png'

assert lowdim_path.exists(), lowdim_path
assert rgb_path.exists(), rgb_path
assert depth_path.exists(), depth_path
assert LEFT_JSON.exists(), LEFT_JSON
assert RIGHT_JSON.exists(), RIGHT_JSON

lowdim = np.load(lowdim_path, allow_pickle=True).item()
rgb = read_rgb(rgb_path)
height, width = rgb.shape[:2]

cam_to_left_base = load_pose_in_link(LEFT_JSON)
cam_to_right_base = load_pose_in_link(RIGHT_JSON)

left_cfg = OmegaConf.load(LEFT_CFG_PATH)
right_cfg = OmegaConf.load(RIGHT_CFG_PATH)

if USE_LOWDM_GRIPPER_FOR_LAST_JOINT:
    left_joint = np.zeros(8, dtype=np.float32)
    right_joint = np.zeros(8, dtype=np.float32)
    if 'gripper_left' in lowdim:
        left_joint[-1] = float(np.asarray(lowdim['gripper_left'])[0])
    if 'gripper_right' in lowdim:
        right_joint[-1] = float(np.asarray(lowdim['gripper_right'])[0])
else:
    left_joint = LEFT_JOINT_MANUAL.copy()
    right_joint = RIGHT_JOINT_MANUAL.copy()

print('scene_path =', scene_path)
print('camera_serial =', camera_serial)
print('rgb_path =', rgb_path)
print('lowdim_path =', lowdim_path)
print('LEFT_JSON =', LEFT_JSON)
print('RIGHT_JSON =', RIGHT_JSON)
print('left_joint =', left_joint)
print('right_joint =', right_joint)
print('cam_to_left_base =\n', cam_to_left_base)
print('cam_to_right_base =\n', cam_to_right_base)
print('intrinsic =\n', intrinsic)


In [ ]:
# Branch A: 严格模拟共享 cam_to_base 分支
cam_to_overall_real_base_from_left = cam_to_left_base @ ROBOT_LEFT_REAL_BASE_TO_REAL_BASE
cam_to_overall_real_base_from_right = cam_to_right_base @ ROBOT_RIGHT_REAL_BASE_TO_REAL_BASE
cam_to_overall_real_base_avg = average_xyz_rot_quat(
    cam_to_overall_real_base_from_left,
    cam_to_overall_real_base_from_right,
    rotation_rep='matrix'
)
cam_to_base_shared = cam_to_overall_real_base_avg @ np.linalg.inv(ROBOT_PREDEFINED_TRANSFORMATION)

renderer_shared = ArmOnlyRobotRenderer(
    left_joint_cfgs=left_cfg,
    right_joint_cfgs=right_cfg,
    cam_to_base=cam_to_base_shared,
    intrinsic=intrinsic,
    urdf_file=ROBOT_INHAND_URDF,
    width=width,
    height=height,
    near_plane=0.01,
    far_plane=100.0,
)
renderer_shared.update_joints(left_joint, right_joint)
render_shared = renderer_shared.render_image()
mask_shared = renderer_shared.render_mask()
overlay_shared = overlay_render(rgb, render_shared, mask_shared)

print('cam_to_overall_real_base_from_left =\n', cam_to_overall_real_base_from_left)
print('cam_to_overall_real_base_from_right =\n', cam_to_overall_real_base_from_right)
print('cam_to_base_shared =\n', cam_to_base_shared)

plt.figure(figsize=(18, 6))
plt.subplot(1, 3, 1)
plt.title('A shared | real rgb')
plt.imshow(rgb)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title('A shared | render')
plt.imshow(render_shared)
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title('A shared | overlay')
plt.imshow(overlay_shared)
plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Branch B: 严格模拟左右独立 cam_to_left_base / cam_to_right_base 分支
renderer_sep = SeparateRobotRenderer(
    left_joint_cfgs=left_cfg,
    right_joint_cfgs=right_cfg,
    cam_to_left_base=cam_to_left_base,
    cam_to_right_base=cam_to_right_base,
    intrinsic=intrinsic,
    width=width,
    height=height,
    urdf_file={
        'left': LEFT_URDF,
        'right': RIGHT_URDF,
    },
)
renderer_sep.update_joints(left_joint, right_joint)
render_sep = renderer_sep.render_image()
mask_sep = renderer_sep.render_mask()
overlay_sep = overlay_render(rgb, render_sep, mask_sep)

print('cam_to_left_base =\n', cam_to_left_base)
print('cam_to_right_base =\n', cam_to_right_base)

plt.figure(figsize=(18, 6))
plt.subplot(1, 3, 1)
plt.title('B separate | real rgb')
plt.imshow(rgb)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title('B separate | render')
plt.imshow(render_sep)
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title('B separate | overlay')
plt.imshow(overlay_sep)
plt.axis('off')
plt.tight_layout()
plt.show()


In [2]:
# Branch C: 把 JSON 先接回 fake robot calib，再让官方 CalibrationInfo/renderer 正常吃（改进版）
from scipy.spatial.transform import Rotation as R
from airexo.calibration.calib_info import CalibrationInfo
from airexo.helpers.constants import ROBOT_LEFT_CAM_TO_TCP, ROBOT_RIGHT_CAM_TO_TCP

def invert_T(T):
    T = np.asarray(T, dtype=np.float64)
    out = np.eye(4, dtype=np.float64)
    out[:3, :3] = T[:3, :3].T
    out[:3, 3] = -T[:3, :3].T @ T[:3, 3]
    return out

def mat_to_pose7_quat(T):
    T = np.asarray(T, dtype=np.float64)
    qxyzw = R.from_matrix(T[:3, :3]).as_quat()
    qx, qy, qz, qw = qxyzw
    return np.array([T[0, 3], T[1, 3], T[2, 3], qw, qx, qy, qz], dtype=np.float32)

# 1) 把 JSON 视作训练/projector 已验证有效的 camera -> left/right real base
cam_to_left_real_base = cam_to_left_base.copy()
cam_to_right_real_base = cam_to_right_base.copy()

# 2) 不再使用 fake calib 里的旧 extrinsics / inhand serial，而是构造一份“自洽”的最简标定
#    思路：令 global / left inhand / right inhand 三者 extrinsics 全相同，这样
#    get_camera_to_robot_left_base() 会退化为：
#       cam_to_left_predefined_base = ROBOT_LEFT_CAM_TO_TCP @ inv(tcp_pose_left)
#    从而只保留我们从 JSON 回填进去的那一段语义。
identity_ext = np.eye(4, dtype=np.float32)
extrinsics_minimal = {
    camera_serial: identity_ext.copy(),
    'json_inhand_left': identity_ext.copy(),
    'json_inhand_right': identity_ext.copy(),
}

cam_to_left_predefined_base = cam_to_left_real_base @ ROBOT_LEFT_REAL_BASE_TO_REAL_BASE @ np.linalg.inv(ROBOT_PREDEFINED_TRANSFORMATION)
cam_to_right_predefined_base = cam_to_right_real_base @ ROBOT_RIGHT_REAL_BASE_TO_REAL_BASE @ np.linalg.inv(ROBOT_PREDEFINED_TRANSFORMATION)

tcp_pose_left_mat = invert_T(cam_to_left_predefined_base) @ ROBOT_LEFT_CAM_TO_TCP
tcp_pose_right_mat = invert_T(cam_to_right_predefined_base) @ ROBOT_RIGHT_CAM_TO_TCP

tcp_pose_left = mat_to_pose7_quat(tcp_pose_left_mat)
tcp_pose_right = mat_to_pose7_quat(tcp_pose_right_mat)

tmp_fake_timestamp = 'tmp_json_backfilled_robot_calib_v2'
tmp_fake_path = CALIB_DIR / f'{tmp_fake_timestamp}.npy'
tmp_fake_calib = {
    'type': 'robot',
    'camera_serials': [camera_serial, 'json_inhand_left', 'json_inhand_right'],
    'camera_serials_global': [camera_serial],
    'camera_serial_inhand_left': 'json_inhand_left',
    'camera_serial_inhand_right': 'json_inhand_right',
    'intrinsics': {
        camera_serial: intrinsic.astype(np.float32),
        'json_inhand_left': intrinsic.astype(np.float32),
        'json_inhand_right': intrinsic.astype(np.float32),
    },
    'extrinsics': extrinsics_minimal,
    'robot_left': {
        'tcp_pose': tcp_pose_left,
        'joint_pos': np.zeros((8,), dtype=np.float32),
        'tcp_vel': np.zeros((6,), dtype=np.float32),
        'joint_vel': np.zeros((8,), dtype=np.float32),
        'force_torque': np.zeros((6,), dtype=np.float32),
    },
    'robot_right': {
        'tcp_pose': tcp_pose_right,
        'joint_pos': np.zeros((8,), dtype=np.float32),
        'tcp_vel': np.zeros((6,), dtype=np.float32),
        'joint_vel': np.zeros((8,), dtype=np.float32),
        'force_torque': np.zeros((6,), dtype=np.float32),
    },
}
np.save(tmp_fake_path, tmp_fake_calib, allow_pickle=True)

tmp_calib_info = CalibrationInfo(calib_path=str(CALIB_DIR), calib_timestamp=tmp_fake_timestamp)
cam_to_base_reconstructed = tmp_calib_info.get_camera_to_base(camera_serial)
cam_to_left_reconstructed = tmp_calib_info.get_camera_to_robot_left_base(camera_serial)
cam_to_right_reconstructed = tmp_calib_info.get_camera_to_robot_right_base(camera_serial)

renderer_c = ArmOnlyRobotRenderer(
    left_joint_cfgs=left_cfg,
    right_joint_cfgs=right_cfg,
    cam_to_base=cam_to_base_reconstructed,
    intrinsic=intrinsic,
    urdf_file=ROBOT_INHAND_URDF,
    width=width,
    height=height,
    near_plane=0.01,
    far_plane=100.0,
)
renderer_c.update_joints(left_joint, right_joint)
render_c = renderer_c.render_image()
mask_c = renderer_c.render_mask()
overlay_c = overlay_render(rgb, render_c, mask_c)

print('tmp_fake_path =', tmp_fake_path)
print('tcp_pose_left =', tcp_pose_left)
print('tcp_pose_right =', tcp_pose_right)
print('cam_to_left_predefined_base =', cam_to_left_predefined_base)
print('cam_to_right_predefined_base =', cam_to_right_predefined_base)
print('cam_to_left_reconstructed =', cam_to_left_reconstructed)
print('cam_to_right_reconstructed =', cam_to_right_reconstructed)
print('cam_to_base_reconstructed =', cam_to_base_reconstructed)
print('cam_to_right_reconstructed =', cam_to_right_reconstructed)
print('cam_to_base_reconstructed =', cam_to_base_reconstructed)

plt.figure(figsize=(18, 6))
plt.subplot(1, 3, 1)
plt.title('C backfilled calib v2 | real rgb')
plt.imshow(rgb)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title('C backfilled calib v2 | render')
plt.imshow(render_c)
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title('C backfilled calib v2 | overlay')
plt.imshow(overlay_c)
plt.axis('off')
plt.tight_layout()
plt.show()


NameError: name 'cam_to_left_base' is not defined